# Chapter 16: Hallucination Detection via Geodesic Distance

**Learning objectives:**
- Treat hallucination as a *response-level* claim routed through the Chapter 8--10 pipeline, not a separate system.
- Use `HallucinationOracle` (a Chapter 15 `OracleBase`) to classify agent output with the **same** verdict vocabulary the rest of the harness uses: `SubVerdict` (Ch9), `TypedVerdict` (Ch10), `OracleVerdict` (Ch15).
- Load the calibrated $\tau_{\text{geo}}$ from `store/calibration.json` instead of hard-coding a threshold, and label raw geodesics with `geodesic_band` as multiples of that calibrated value.
- Produce adversarial probes via `HallucinationGenerator` and aggregate results with the Chapter 10 `SeverityClassifier`, which tags every hallucinated claim CRITICAL through the `FailureCode.HALLUCINATION_DETECTED` base severity.

---
## Preamble

The first fifteen chapters built a consistent **typed claim pipeline**:

1. Chapter 8 extracts structured claims from free-text responses (`ClaimCompiler`).
2. Chapter 9 verifies each claim on the GMS manifold (`GraphVerifier`, `SubVerdict`).
3. Chapter 10 aggregates sub-verdicts into a `TypedVerdict` and classifies failures with `SeverityClassifier` keyed on `FailureCode`.
4. Chapter 15 exposes oracles (`OracleBase` $\to$ `OracleVerdict`) for use inside `TieredReleaseGate`.

Chapter 16 does **not** introduce a parallel assessment type. It reuses this pipeline for one specific failure mode --- ungrounded generation --- by:

1. Configuring a `GraphVerifier` with the calibrated $\tau_{\text{geo}}$ already produced by Chapter 8's `GMSJudge.calibrate()`.
2. Routing free-text response claims through it with `failure_code_on_exceed=FailureCode.HALLUCINATION_DETECTED`, so the Chapter 10 severity map applies CRITICAL automatically.
3. Wrapping the same verifier in the Chapter 15 oracle interface so hallucination participates in tiered release gates on equal footing with `ExactOutcomeOracle` and `TraceOracle`.

The adversarial `HallucinationGenerator` targets three geometric weak points of the store: boundary entities (weakly anchored on $S^{m-1}$), near-miss entities (plausible but invented), and open-ended elaboration (forced over-extension).

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
from importlib.resources import files
# Load the Chapter 4 store
from knowlytix.harness.doe import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.doe").joinpath("tests/model_risk_assessment.md")),
    stores_dir="monograph/lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=200,
    device=str(DEVICE),
)
bench = DOEGMSBenchmark(config)
store = bench.ingest()
print(f"Store: {store.stats()['entities']} entities, "
      f"{store.stats()['triples']} triples, "
      f"{store.stats()['enm_entries']} ENM entries")

---

## 16.1 Reusing the Chapter 8--10 Pipeline

`HallucinationOracle` owns no verification logic of its own. Its constructor:

1. Loads $\tau_{\text{geo}}$ from `store/calibration.json` via `_load_calibrated_thresholds` (the same function every Chapter 15 oracle uses).
2. Instantiates a standard `GraphVerifier` (Chapter 9) with `plausibility_threshold=$\tau_{\text{geo}}$` and `failure_code_on_exceed=FailureCode.HALLUCINATION_DETECTED`.

So the `SubVerdict` emitted by `assess_claim` is the exact same dataclass returned by `GraphVerifier.verify` for any `FactClaim` in Chapter 9 --- hallucination just uses a different failure code so `SeverityClassifier` applies the CRITICAL band.

In [ ]:
from knowlytix.harness.doe import HallucinationOracle, geodesic_band

# tau_geodesic=None (default) reads store/calibration.json.
oracle = HallucinationOracle(store=store)

print(f"tau_geodesic (calibrated): {oracle.tau_geodesic:.4f}")
print(f"calibration source:       {oracle._calibration_source}")
print(f"failure code on exceed:   "
      f"{oracle._graph_verifier.failure_code_on_exceed.value}")
print()
print("Underlying verifier is a standard Chapter 9 GraphVerifier:")
print(f"  type: {type(oracle._graph_verifier).__name__}")
print(f"  plausibility_threshold: "
      f"{oracle._graph_verifier.plausibility_threshold:.4f}")

---

## 16.2 Claim-Level Entry Point --- `SubVerdict`

`oracle.assess_claim(head, relation, tail)` calls `GraphVerifier.verify` under the hood and returns the Chapter 9 `SubVerdict`. Missing entities and unscorable relations are promoted to the geodesic ceiling (2.0) and the failure code is normalized to `HALLUCINATION_DETECTED` regardless of which structural failure `GraphVerifier` observed.

In [4]:
# A known triple from the store --- should pass.
known = oracle.assess_claim("MDL-001", "in_section", "performance_metrics")
print("Known triple:")
print(f"  passed:        {known.passed}")
print(f"  raw_score:     {known.raw_score:.4f}  (geodesic)")
print(f"  band:          {geodesic_band(known.raw_score, oracle.tau_geodesic)}")
print(f"  failure:       {known.failure}")
print()

# A fabricated triple --- entity not in the store.
fab = oracle.assess_claim("MDL-999", "has_algorithm", "QuantumBoost")
print("Fabricated triple:")
print(f"  passed:        {fab.passed}")
print(f"  raw_score:     {fab.raw_score:.4f}  (geodesic; ceiling if OOV)")
print(f"  band:          {geodesic_band(fab.raw_score, oracle.tau_geodesic)}")
if fab.failure:
    print(f"  failure code:  {fab.failure.code.value}")
    print(f"  severity:      {fab.failure.severity.value}")

Known triple:
  passed:        True
  raw_score:     1.3398  (geodesic)
  band:          grounded
  failure:       None

Fabricated triple:
  passed:        False
  raw_score:     2.0000  (geodesic; ceiling if OOV)
  band:          fabrication
  failure code:  hallucination_detected
  severity:      error


---

## 16.3 Batch Entry Point --- `TypedVerdict`

`oracle.assess_triples([...])` returns a Chapter 10 `TypedVerdict` via `TypedVerdict.from_sub_verdicts`. The returned object carries the full audit trail (per-claim `SubVerdict` list and `FailureAttribution` records) that `SeverityClassifier` expects --- no bespoke aggregation, no separate report type.

In [ ]:
from knowlytix.harness.doe.verdict import TypedVerdict
from knowlytix.harness.doe.severity import SeverityClassifier

mixed_claims = [
    ("MDL-001", "in_section", "performance_metrics"),  # grounded
    ("MDL-002", "in_section", "3.0"),                  # corrupted
    ("MDL-999", "has_algorithm", "QuantumBoost"),      # fabrication
    ("FraudNet-LSTM", "invented_by", "deep_learning"), # fabrication
]

verdict = oracle.assess_triples(mixed_claims, run_id="ch16_demo", question_id="q_mixed")
assert isinstance(verdict, TypedVerdict)

print("=== TypedVerdict ===")
print(f"  passed:       {verdict.correct}")
print(f"  n sub-verdicts: {len(verdict.sub_verdicts)}")
print(f"  n failures:     {len(verdict.failures)}")
print()

for sv in verdict.sub_verdicts:
    band = geodesic_band(sv.raw_score, oracle.tau_geodesic)
    code_str = sv.failure.code.value if sv.failure else "-"
    print(f"  passed={sv.passed}  geodesic={sv.raw_score:.3f}  "
          f"band={band:<14s}  code={code_str}")

In [6]:
# Every failure routes through the Chapter 10 severity pipeline.
classifier = SeverityClassifier()
assessments = classifier.classify_all(verdict.failures)
counts = classifier.count_by_level(assessments)

print("=== SeverityClassifier on the TypedVerdict ===")
print(f"  n failures classified: {len(assessments)}")
for level, n in counts.items():
    if n:
        print(f"  {level.value:<10s} {n}")
print()
for f, a in zip(verdict.failures, assessments):
    print(f"  {f.code.value:<28s} level={a.level.value:<10s} "
          f"composite={a.composite_score:.2f}")
print()
print("Note: `hallucination_detected` carries CRITICAL base severity in the Ch10 "
      "failure-code map, so every failing claim bubbles up as CRITICAL regardless "
      "of which structural signal `GraphVerifier` observed.")

=== SeverityClassifier on the TypedVerdict ===
  n failures classified: 3
  critical   3

  hallucination_detected       level=critical   composite=0.81
  hallucination_detected       level=critical   composite=0.80
  hallucination_detected       level=critical   composite=0.80

Note: `hallucination_detected` carries CRITICAL base severity in the Ch10 failure-code map, so every failing claim bubbles up as CRITICAL regardless of which structural signal `GraphVerifier` observed.


---

## 16.4 Response-Level Entry Point --- `OracleVerdict`

`oracle.evaluate(scenario, trace)` is the Chapter 15 oracle interface. It builds the canonical `(ground_truth, "has_answer", raw_output)` triple from the trace and calls `assess_claim`. This is the path `TieredReleaseGate` uses --- hallucination plugs into tiered release on equal footing with `ExactOutcomeOracle` and `TraceOracle`.

In [ ]:
from knowlytix.harness.doe.scenarios import Scenario
from knowlytix.harness.doe.adapters import SUTTrace

scenario = Scenario(
    scenario_id="halluc_demo",
    question_text="What is the algorithm behind MDL-001?",
    ground_truth="MDL-001",
    metadata={},
)

grounded_trace = SUTTrace(raw_output="XGBoost")
ungrounded_trace = SUTTrace(raw_output="QuantumBoost ensemble")

v1 = oracle.evaluate(scenario, grounded_trace)
v2 = oracle.evaluate(scenario, ungrounded_trace)

for label, v in [("grounded response", v1), ("ungrounded response", v2)]:
    print(f"=== {label} ===")
    print(f"  oracle_type: {v.oracle_type}")
    print(f"  passed:      {v.passed}")
    print(f"  confidence:  {v.confidence:.3f}")
    print(f"  detail:      {v.detail}")
    print(f"  raw_scores:  {v.raw_scores}")
    print()

---

## 16.5 Severity Bands Anchored on $\tau_{\text{geo}}$

`geodesic_band(d, tau_geo)` is a presentation utility only:

Let $\Delta = (\text{ceiling} - \tau) / 3$ with $\text{ceiling} = 2.0$ (the OOV geodesic cap). The three super-threshold bands then split the interval $(\tau, \text{ceiling}]$ equally:

| Geodesic | Band |
|---|---|
| $d \le \tau$ | `grounded` |
| $\tau < d \le \tau + \Delta$ | `embellishment` |
| $\tau + \Delta < d \le \tau + 2\Delta$ | `distortion` |
| $d > \tau + 2\Delta$ (incl. OOV ceiling) | `fabrication` |

Bands are anchored on $\tau_{\text{geo}}$ and the manifold ceiling, so recalibrating the judge shifts every band automatically and no band is ever unreachable --- even when $\tau$ lands close to the ceiling. **Governance severity is not read off this label** --- it comes from `SeverityClassifier` on the underlying `FailureCode.HALLUCINATION_DETECTED`.

In [8]:
# Scan 100 known triples from the store and see the band distribution.
import random
random.seed(42)
triples = list(store.doc_graph.triples)
random.shuffle(triples)

known_bands = Counter()
known_geos = []
for h, r, t in triples[:100]:
    sv = oracle.assess_claim(h, r, t)
    known_geos.append(sv.raw_score)
    known_bands[geodesic_band(sv.raw_score, oracle.tau_geodesic)] += 1

# And a handful of fabricated triples.
fabricated_claims = [
    ("MDL-001", "has_algorithm", "QuantumBoosting"),
    ("CreditScore-XGB", "has_detail", "Alan_Turing"),
    ("NonexistentModel", "has_risk_tier", "Ultra-Critical"),
    ("FraudNet-LSTM", "invented_by", "deep_learning_lab"),
    ("MDL-099", "has_accuracy", "0.9999"),
]
fab_bands = Counter()
fab_geos = []
for h, r, t in fabricated_claims:
    sv = oracle.assess_claim(h, r, t)
    fab_geos.append(sv.raw_score)
    fab_bands[geodesic_band(sv.raw_score, oracle.tau_geodesic)] += 1

print(f"tau_geodesic = {oracle.tau_geodesic:.4f}")
print(f"{'Band':<16s} {'Known (100)':>12s} {'Fabricated (5)':>15s}")
print("-" * 47)
for b in ["grounded", "embellishment", "distortion", "fabrication"]:
    print(f"{b:<16s} {known_bands.get(b, 0):>12d} {fab_bands.get(b, 0):>15d}")
print()
print(f"Known   geodesic: mean={np.mean(known_geos):.3f}  max={np.max(known_geos):.3f}")
print(f"Fabric. geodesic: mean={np.mean(fab_geos):.3f}  max={np.max(fab_geos):.3f}")

tau_geodesic = 1.3845
Band              Known (100)  Fabricated (5)
-----------------------------------------------
grounded                   97               0
embellishment               3               0
distortion                  0               0
fabrication                 0               5

Known   geodesic: mean=1.204  max=1.411
Fabric. geodesic: mean=2.000  max=2.000


---

## 16.6 Lab --- Threshold Sensitivity

Recalibration shifts $\tau_{\text{geo}}$. The bands move with it; the band logic does not need editing. To illustrate, we override `tau_geodesic` for a sweep and compare the fraction of known vs. fabricated claims flagged.

In [9]:
tau_values = [0.5, 0.7, 1.0, 1.5, 2.0]
print(f"{'tau':>6s} {'known flagged':>14s} {'fab flagged':>12s}")
print("-" * 36)
for tau in tau_values:
    probe = HallucinationOracle(store=store, tau_geodesic=tau)
    n_known = sum(1 for h, r, t in triples[:100]
                  if not probe.assess_claim(h, r, t).passed)
    n_fab = sum(1 for h, r, t in fabricated_claims
                if not probe.assess_claim(h, r, t).passed)
    print(f"{tau:>6.2f} {n_known:>14d}/100 {n_fab:>8d}/{len(fabricated_claims)}")

   tau  known flagged  fab flagged
------------------------------------


  0.50            100/100        5/5


  0.70            100/100        5/5


  1.00             94/100        5/5


  1.50              0/100        5/5


  2.00              0/100        5/5


---

## 16.7 `HallucinationGenerator` --- Geometric Probes

The generator targets three situations where hallucination is most likely:

1. **Boundary entities** (few triples, weakly anchored on $S^{m-1}$).
2. **Near-miss entities** (nonexistent names similar to real ones, verified via `fuzzy_match_entity`).
3. **Open-ended elaboration** (force the agent to elaborate beyond its grounding).

In [ ]:
from knowlytix.harness.doe import HallucinationGenerator

gen = HallucinationGenerator()
probes = gen.generate(store)

strategies = Counter(p.metadata["strategy"] for p in probes)
print(f"Generated {len(probes)} hallucination probes")
for s, n in sorted(strategies.items()):
    print(f"  {s:<18s} {n}")
print()
print(f"{'Strategy':<20s} {'QID':<26s} {'Target'}")
print("-" * 72)
for p in probes[:6]:
    target = p.metadata.get("entity") or p.metadata.get("fake_entity") or "-"
    print(f"{p.metadata['strategy']:<20s} {p.qid:<26s} {target}")

---

## 16.8 Lab --- End-to-End Hallucination Detection on a Mock Agent

Run the full DOE benchmark, route every trace through `oracle.evaluate`, and aggregate with `SeverityClassifier`. This exercises the same code path a production release gate uses.

> **Caveat --- the `has_answer` shortcut.** `oracle.evaluate` treats `raw_output` as a single claim `(ground_truth, "has_answer", raw_output)`. That shortcut is useful for a quick end-to-end gate, but it is a pessimistic approximation: a mock agent that returns opaque numeric answers (e.g. `"0.85"`) produces triples that are OOV under `has_answer` in any store, so they saturate at the geodesic ceiling. For richer behavior, pre-compile the response with `ClaimCompiler` and call `assess_triples`; the sections above show that path.

In [ ]:
from importlib.resources import files
from knowlytix.harness.doe.adapters import AgentAdapter
from knowlytix.harness.fixtures import FullMockHarness

adapter = AgentAdapter(harness=FullMockHarness(recovery_mode="fallback"))
context = open(str(files("knowlytix.harness.doe").joinpath("tests/model_risk_assessment.md"))).read()

# Use a modest subset for the notebook; scale as needed in lab runs.
doe_questions = bench.generate_questions()[:120]

verdicts = []
for dq in doe_questions:
    trace = adapter.evaluate(
        question=dq.question.natural_language,
        context=context,
    )
    scenario = Scenario(
        scenario_id=dq.question.qid,
        question_text=dq.question.natural_language,
        ground_truth=str(dq.question.ground_truth) if dq.question.ground_truth is not None else "answer",
        metadata=dict(dq.question.metadata),
    )
    verdicts.append(oracle.evaluate(scenario, trace))

passed = sum(1 for v in verdicts if v.passed)
print(f"{len(verdicts)} responses scored")
print(f"  grounded (passed):   {passed} ({100*passed/len(verdicts):.1f}%)")
print(f"  hallucinated:        {len(verdicts)-passed} "
      f"({100*(len(verdicts)-passed)/len(verdicts):.1f}%)")

# Band distribution across all responses.
bands = Counter()
for v in verdicts:
    g = v.raw_scores.get("geodesic", 0.0)
    bands[geodesic_band(g, oracle.tau_geodesic)] += 1
print()
print("Band distribution:")
for b in ["grounded", "embellishment", "distortion", "fabrication"]:
    print(f"  {b:<15s} {bands.get(b, 0)}")

---

## 16.9 Exercises

**Exercise 16.1** --- Re-run Section 16.5 with the override `tau_geodesic=0.5`. How many known triples now land in `embellishment` or worse? What does that tell you about over-tightening the calibrated threshold?

**Exercise 16.2** --- Take the first five probes from Section 16.7 and hand-craft two responses per probe: one grounded in the document, one fabricated. Pass each through `oracle.evaluate` and confirm that the `OracleVerdict.passed` flag flips as expected.

**Exercise 16.3** --- Combine Section 16.8's verdicts with the Chapter 15 `CompositeOracle` so hallucination participates in the same release gate as `ExactOutcomeOracle`. Observe how `SeverityClassifier` promotes any hallucinated claim to CRITICAL regardless of whether the exact-outcome oracle passed.

---

## Summary

Chapter 16 is a **consistency chapter**, not a new subsystem:

1. **Hallucination is one failure code.** `FailureCode.HALLUCINATION_DETECTED` sits in the Chapter 10 taxonomy; the severity pipeline already knows what to do with it.

2. **No parallel verdict types.** `HallucinationOracle.assess_claim` returns `SubVerdict` (Ch9); `assess_triples` returns `TypedVerdict` (Ch10); `evaluate` returns `OracleVerdict` (Ch15). Every type is reused.

3. **Calibration, not constants.** The oracle loads $\tau_{\text{geo}}$ from `store/calibration.json` via the same helper every other Chapter 15 oracle uses. Severity bands are multiples of $\tau_{\text{geo}}$, not hard-coded.

4. **`GraphVerifier` is the authority.** The oracle delegates to a standard Chapter 9 `GraphVerifier` configured with `failure_code_on_exceed=HALLUCINATION_DETECTED`, so the verification signal is identical to any other fact check in the harness --- only the downstream label differs.

5. **Probes target geometric weak points.** `HallucinationGenerator` stresses three situations where the manifold has little support: boundary entities, near-miss entities and open-ended elaboration.

The harness now has an end-to-end hallucination pathway that plugs straight into `TieredReleaseGate` without introducing a single new dataclass.